# PROC GAMによる非線形小売需要曲線のモデル化


## エグゼクティブサマリー

本ノートブックは**PROC GAM**を用いて、週次の食料品販売数量を、棚価格・店舗気温(季節性の代理変数)・プロモーション支出のなめらかな非線形関数としてモデル化し、パラメトリックなプロモーションフラグ効果も併せて推定する。一般化加法モデルにより、カテゴリーマネージャーは線形回帰では見逃してしまう真の曲線的な価格弾力性と季節需要の形状を復元でき、より精緻な価格設定・プロモーション判断を支援する。


## データソース

| データセット | 行数 | 粒度 | 主要変数 | 説明 |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | 週 | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | 1つの主力食料品店における100週連続(季節サイクル約2回分)の合成週次POSデータ。`call streaminit(20250531)`と`rand()`によりインラインで生成される。週次販売数量はポアソン需要過程に従い、その平均は指数型の価格応答曲線、72F付近でピークを迎える二次関数の気温・季節性効果、そして凹型(平方根)のプロモーション支出リフト、さらに離散的なプロモーション週フラグによって決定される。 |


# PROC GAMによる非線形小売需要曲線のモデル化

小売需要が価格・天候・プロモーション支出に対して直線的に反応することはめったにない。定番商品のわずかな値下げはほとんど数量を動かさないことがある一方、心理的な価格帯を超えると急激な増加を引き起こすことがある。天候主導の需要は快適な中間帯でピークを迎え、両極端で落ち込む。プロモーションのリフトは支出が増えるにつれて逓減する。

**PROC GAM**(一般化加法モデル)は各要因をそれぞれ独自の平滑スプライン項で当てはめるため、厳格な線形の仮定ではなく——データそのものが各需要曲線の形状を決定する。ここでは、1つの主力食料品店における100週連続の週次販売数量を、パラメトリックなプロモーションフラグと、価格・気温・プロモーション支出に対する平滑化スプラインを組み合わせて、ポアソン応答のもとでモデル化する。


## ステップ1 — 合成の週次販売系列を生成する

1つの主力店舗について、100週連続(季節サイクル約2回分)のPOSデータをシミュレートする。データ生成過程は意図的に非線形にしてあり、GAMが現実的な形状を復元できることを確認できるようにしている:

- **Price(価格)**は指数型の応答曲線(`exp(-1.1 * Price)`)を通じて数量を左右し、価格が下がるほど需要は急激に上昇する。
- **Temperature(気温)**は72F付近に二次関数のピークを持つ季節性の代理変数として働き、快適さに駆動される来店行動をモデル化する。
- **PromoSpend(プロモーション支出)**は凹型の平方根リフト(逓減効果)をもたらす。
- 離散的な**Promotion(プロモーション)**フラグは、プロモーション実施週にパラメトリックなステップ効果を加える。

週次の`Units`(販売数量)はポアソン分布から生成され、単位販売数のカウント特性に整合する。


In [1]:
データ store_sales;
   呼出 streaminit(20250531);
   長さ Promotion $3;
   繰返 Week = 1 から 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      もし rand("uniform") < 0.28 なら 繰返;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      終了;
      他 繰返;
         Promotion  = "No";
         PromoSpend = 0;
      終了;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      出力;
   終了;
実行;



NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## ステップ2 — シミュレートしたデータのプロファイル

簡単な`PROC MEANS`により、モデル化前に各変数が妥当な小売レンジに収まっていることを確認する:販売数量は非負の整数、価格は数ドル前後に集中し、気温は季節サイクルを一巡し、プロモーション支出は非プロモーション週にはゼロとなる。


In [2]:
処理 平均 データ=store_sales n mean MIN MAX maxdec=2;
   変数 Units Price Temperature PromoSpend;
   見出 Units="販売数量" Price="価格" Temperature="気温" PromoSpend="プロモ支出";
実行;


                                                  The MEANS Procedure

 Variable     Label                   N           Mean     Minimum     Maximum
 -----------------------------------------------------------------------------
 Units        販売数量                  100           7.03        0.00      103.00
 Price        価格                    100           3.15        2.81        3.48
 Temperature  気温                    100          55.50       22.72       83.49
 PromoSpend   プロモ支出                 100         128.76        0.00      779.00
 -----------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ3 — 完全な加法的需要モデルの当てはめ

このコアモデルは以下を組み合わせる:

- `param(Promotion)` — `CLASS`ステートメントで宣言される、プロモーション週指標に対するパラメトリック(線形)効果。
- `spline(Price, df=4)` — 曲線的な価格応答を捉える3次平滑化スプライン。
- `spline(Temperature, df=4)` — なめらかな季節性曲線。
- `spline(PromoSpend, df=3)` — 逓減するプロモーションリフト。

販売数量はカウントデータであるため、`dist=poisson`(ログリンク)を指定する。`method=gcv`は、明示的な自由度の指定なしに、一般化交差検証によって各成分の平滑さを決定させる。`OUTPUT`ステートメントは観測ごとの予測値と残差を`gam_fit`に書き出す。

このプロシジャは、**Null Deviance 2041.12**に対して**Deviance 43.59**を報告する——4つの平滑・パラメトリック要因が、定数項のみのモデルが残すばらつきのほぼすべてを説明していることを示す——また**AICは254.61**である。パラメトリックな`PROMOTIONYES`推定値は正(対数尺度で+0.41)であり、プロモーションによる数量リフトを裏づけている。価格スプラインは強い負の線形トレンド(−1.71)を持ち、これは右下がりの需要の特徴である。


In [3]:
処理 gam データ=store_sales PLOTS=components(CLM commonaxes);
   分類 Promotion;
   模型 Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   出力 out=gam_fit predicted residual;
   見出 Units="販売数量" Promotion="プロモーション" Price="価格"
         Temperature="気温" PromoSpend="プロモ支出";
実行;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     販売数量
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(価格)                       4.0000         4.0


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## ステップ4 — 価格と季節性に絞ったモデル

より絞り込んだ価格レビューのため、意思決定に最も関連の深い2つの平滑要因——価格と気温——のみで再度当てはめる。価格には、心理的な価格帯付近の折れ曲がりを解決できるよう追加の柔軟性(`df=5`)を与える。プロモーションフラグはパラメトリック制御としてそのまま残す。

プロモーション支出を除くと、**Devianceは62.80**、**AICは269.82**まで上昇し、いずれもフルモデルの43.59および254.61より高くなる。パラメトリックな`PROMOTIONYES`項も、支出スプラインがそれを吸収しなくなったため、ここではより多くのプロモーション信号を吸収する(+0.41に対して+1.73)。価格スプラインは負のトレンド(−1.51)を維持しており、コアとなる需要のストーリーは仕様間で安定している。


In [4]:
処理 gam データ=store_sales PLOTS=components(CLM);
   分類 Promotion;
   模型 Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   見出 Units="販売数量" Promotion="プロモーション" Price="価格"
         Temperature="気温";
実行;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     販売数量
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(価格)                       5.0000         5.0000
Spline(気温)                       4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## 結果の解釈

**Regression Model Analysis**(回帰モデル分析)テーブルは、各成分内の線形トレンドとパラメトリックなプロモーション効果を報告する。正の`PROMOTIONYES`係数(フルモデルで+0.41、絞り込んだモデルで+1.73)は期待どおりのプロモーション数量リフトを裏づけ、価格スプラインの負の線形トレンド(−1.71および−1.51)は典型的な右下がりの需要を反映している。気温スプラインの小さな正の線形項(+0.03)は想定どおりである:その本質的な物語は72Fの快適ピーク付近の曲率にあり、単一の線形係数では要約できない。

**Smoothing Model Analysis**(平滑化モデル分析)テーブルは、各スプラインが消費する自由度を報告する。価格と気温はそれぞれ実効自由度4(絞り込んだモデルでは価格が5)を、プロモーション支出は3を消費する——いずれも直線が使う単一の自由度を大きく上回っており、これこそ線形回帰がこれらの曲線的な関係を見逃してしまう理由である。

**Fit Statistics**(適合統計量)により、2つの仕様を直接比較できる。フルの4要因モデルはDeviance 43.59、AIC 254.61を示し、絞り込んだモデルの62.80および269.82と対比される。両基準ともフルモデルを支持しており、プロモーション支出とプロモーションフラグが価格・気温だけでは得られない説明力を追加していることを示す。Null Deviance 2041.12と比較すると、両モデルとも需要のばらつきの大部分を捉えている。

これらのテーブルを合わせて読むと、カテゴリーマネージャーに定量的でデータ駆動型の需要ストーリーが提供される:値下げの深さを決める急峻な価格応答、季節的な気温ウィンドウ、そして逓減するプロモーション支出効果——単一の線形弾力性推定よりもはるかに鋭い指針である。(PROC GAMは`plots=components`も受け付け、各平滑項の部分予測曲線を描画できる。上記の数値による成分テーブルは、それらの曲線が導出される元データである。)
